In [ ]:
# Collection, scoring and aggregation live in ../pipeline.py; what makes this
# analysis different lives in config.py next to this notebook.
# Reads only the 0_raw snapshot, so this runs on a cluster with no scratch data.
# See ../../README.md for the snapshot and provenance model.
import sys
sys.path.insert(0, '..')   # pipeline.py
sys.path.insert(0, '.')    # config.py, if the kernel did not add it
import pipeline as pl
from config import CONFIG

# No seed is selected. Every (seed, fold) run is kept and reduced
#   5 folds -> mean -> 1 value per seed -> mean +- SD across 3 seeds
# writes per_run_results.csv, seed_level_results.csv,
#        representation_level_results.csv, coverage.csv
per_run, seed_level, rep_level = pl.run(CONFIG)
rep_level

In [ ]:
# The seeds x folds grid should be complete. Any row here is a hole in it, and
# every number downstream of it rests on fewer runs than the Methods claim.
import pandas as pd
cov = pd.read_csv('coverage.csv')
print(f'{len(cov)} incomplete (model, group, seed) cells')
cov

In [ ]:
# What the SD is made of: three seed-level values per model, each a mean over
# five folds. Worth a look before quoting mean +- SD - a model whose seeds
# disagree by more than the gap to its neighbour is not distinguishable from it.
import pandas as pd
keys = ['model'] + [k for k in ('dir', 'sero', 'group') if k in rep_level.columns]
s = seed_level.pivot_table(index=keys, columns='seed', values='roc_auc')
s = s.join(rep_level.set_index(keys)[['roc_auc', 'roc_auc_sd']]
                    .rename(columns={'roc_auc': 'mean', 'roc_auc_sd': 'sd'}))
s.sort_values('mean', ascending=False).round(4)